# Key Python Concepts from the Autocorrect Assignment
### NLP with Probabilistic Models — DeepLearning.AI

This notebook covers every Python pattern you encountered in the Autocorrect assignment, with explanations and examples you can run and experiment with.

---
## 1. `dict.get(key, default)`

Safely look up a key in a dictionary. If the key doesn't exist, return a default value instead of raising a `KeyError`.

**Syntax:** `my_dict.get(key, default_value)`

**Used in:** `get_count()`, `get_probs()`, `get_corrections()`

In [ ]:
word_count = {'the': 5, 'cat': 2, 'sat': 1}

# Key exists → returns the value
print(word_count.get('cat', 0))      # 2

# Key doesn't exist → returns the default (0)
print(word_count.get('dragon', 0))   # 0

# Without .get() this would raise a KeyError:
# print(word_count['dragon'])  # KeyError!

**Pattern used in the assignment — building word counts:**

In [ ]:
word_l = ['i', 'am', 'happy', 'because', 'i', 'am', 'learning']

word_count_dict = {}
for word in word_l:
    word_count_dict[word] = word_count_dict.get(word, 0) + 1

print(word_count_dict)
# {'i': 2, 'am': 2, 'happy': 1, 'because': 1, 'learning': 1}

{'i': 3, 'am': 3, 'happy': 2, 'because': 2, 'learning': 2}


---
## 2. Dictionary Comprehension with `.items()`

A compact way to create a new dictionary by transforming an existing one.

**Syntax:** `{key: expression for key, value in dict.items()}`

**Used in:** `get_probs()`

In [6]:
word_count_dict = {'i': 2, 'am': 2, 'happy': 1, 'because': 1, 'learning': 1}

# .items() gives you both key and value at once
for word, count in word_count_dict.items():
    print(f"word: {word}, count: {count}")

word: i, count: 2
word: am, count: 2
word: happy, count: 1
word: because, count: 1
word: learning, count: 1


In [7]:
# Dictionary comprehension to compute probabilities
word_sum = sum(word_count_dict.values())  # total words = 7

probs = {word: count / word_sum for word, count in word_count_dict.items()}
print(probs)
# {'i': 0.2857, 'am': 0.2857, 'happy': 0.1428, ...}

{'i': 0.2857142857142857, 'am': 0.2857142857142857, 'happy': 0.14285714285714285, 'because': 0.14285714285714285, 'learning': 0.14285714285714285}


---
## 3. The Split Pattern

The foundation of all four edit operations. Split a word at every possible position into a `(Left, Right)` tuple.

**Used in:** `delete_letter()`, `insert_letter()`, `switch_letter()`, `replace_letter()`

In [ ]:
word = 'cat'

split_l = [(word[:i], word[i:]) for i in range(len(word) + 1)]
print(split_l)
# [('', 'cat'), ('c', 'at'), ('ca', 't'), ('cat', '')]

Once you have splits, all four operations follow the same structure — just change what you do with `L` and `R`:

| Operation | Formula | What it does |
|-----------|---------|---------------|
| Delete | `L + R[1:]` | Removes first char of R |
| Insert | `L + letter + R` | Adds a letter between L and R |
| Swap | `L + R[1] + R[0] + R[2:]` | Swaps first two chars of R |
| Replace | `L + letter + R[1:]` | Replaces first char of R |

In [ ]:
letters = 'abcdefghijklmnopqrstuvwxyz'
word = 'cat'
split_l = [(word[:i], word[i:]) for i in range(len(word) + 1)]

# Delete
delete_l = [L + R[1:] for L, R in split_l if R]
print('delete:', delete_l)

# Insert
insert_l = [L + c + R for L, R in split_l for c in letters]
print('insert (first 5):', insert_l[:5])

# Swap
swap_l = [L + R[1] + R[0] + R[2:] for L, R in split_l if len(R) >= 2]
print('swap:', swap_l)

# Replace
replace_l = [L + c + R[1:] for L, R in split_l if R for c in letters]
replace_l = [w for w in replace_l if w != word]  # remove original word
print('replace (first 5):', replace_l[:5])

---
## 4. Set Operations — `|`, `&`, `|=`

Sets automatically eliminate duplicates. Three operators you used:

| Operator | Name | Meaning |
|----------|------|---------|
| `\|` | Union | All elements from both sets |
| `&` | Intersection | Only elements in both sets |
| `\|=` | Union-assign | Merge another set into this one |

**Used in:** `edit_one_letter()`, `edit_two_letters()`, `get_corrections()`

In [ ]:
a = {1, 2, 3}
b = {3, 4, 5}

print(a | b)   # Union:        {1, 2, 3, 4, 5}
print(a & b)   # Intersection: {3}

# |= merges in place (like += for lists)
result = set()
result |= {1, 2}
result |= {2, 3}  # 2 appears in both but is only stored once
print(result)   # {1, 2, 3}

In [ ]:
# How |= was used in edit_two_letters:
# For each word from edit_one, apply edit_one again and union the results

vocab = {'days', 'dye', 'cat', 'car', 'bar'}

# Filter candidates to only real vocab words using &
candidates = {'dyas', 'days', 'xyz', 'dye'}
valid = candidates & vocab
print(valid)  # {'days', 'dye'}

---
## 5. `lambda` Functions

A one-line anonymous function. Used when you need a simple function but don't want to define it with `def`.

**Syntax:** `lambda arguments: expression`

**Used in:** `get_corrections()` inside `sorted()`

In [ ]:
# These two are identical:

# Regular function
def double(x):
    return x * 2

# Lambda equivalent
double_lambda = lambda x: x * 2

print(double(5))         # 10
print(double_lambda(5))  # 10

---
## 6. `sorted()` with a `key`

Sort any iterable by a custom criterion using the `key` argument.

**Syntax:** `sorted(iterable, key=function, reverse=True/False)`

**Used in:** `get_corrections()`

In [ ]:
probs = {'dye': 0.00002, 'days': 0.00041, 'xyz': 0.0}
suggestions = ['dye', 'days', 'xyz']

# Sort by probability, highest first
sorted_suggestions = sorted(suggestions, key=lambda w: probs.get(w, 0), reverse=True)
print(sorted_suggestions)
# ['days', 'dye', 'xyz']
# 0.00041 > 0.00002 > 0.0

In [ ]:
# More examples of sorted() with key
words = ['banana', 'fig', 'apple', 'kiwi']

# Sort by word length
print(sorted(words, key=lambda w: len(w)))
# ['fig', 'kiwi', 'apple', 'banana']

# Sort by last character
print(sorted(words, key=lambda w: w[-1]))
# ['banana', 'apple', 'fig', 'kiwi']

---
## 7. Short-Circuit Evaluation with `or`

Python's `or` stops evaluating as soon as it finds a non-empty (truthy) value. This lets you build priority chains cleanly.

**Used in:** `get_corrections()` — priority logic for suggestions

In [ ]:
# or returns the first truthy value, or the last value if all are falsy
print([] or ['a', 'b'])       # ['a', 'b']  — first list empty, returns second
print(['x'] or ['a', 'b'])    # ['x']       — first list non-empty, stops here
print([] or [] or ['last'])   # ['last']    — keeps going until non-empty

In [ ]:
# How it was used in get_corrections — priority chain:
vocab = {'days', 'dye', 'play', 'stay'}
word = 'dys'

# Simulated edit results
one_edits = {'dys', 'days', 'xyz'}   # some valid, some not
two_edits = {'dye', 'abc', 'def'}

suggestions = (
    [word] if word in vocab                              # level 0: word itself
    else [w for w in one_edits if w in vocab]            # level 1: one edit
    or [w for w in two_edits if w in vocab]              # level 2: two edits
    or [word]                                            # fallback: original
)

print(suggestions)  # ['days'] — found at level 1, level 2 never evaluated

---
## 8. Dynamic Programming — Minimum Edit Distance

Build a matrix where each cell `D[i][j]` = minimum edits to convert `source[:i]` into `target[:j]`.

**Three moves:**
- **Down** (delete): `D[i-1][j] + del_cost`
- **Right** (insert): `D[i][j-1] + ins_cost`  
- **Diagonal** (replace/match): `D[i-1][j-1] + r_cost` (r_cost=0 if chars match)

**Used in:** `min_edit_distance()`

In [ ]:
import numpy as np
import pandas as pd

def min_edit_distance(source, target, ins_cost=1, del_cost=1, rep_cost=2):
    m = len(source)
    n = len(target)
    D = np.zeros((m+1, n+1), dtype=int)

    # Fill column 0: cost of deleting all source chars
    for row in range(1, m+1):
        D[row, 0] = row * del_cost

    # Fill row 0: cost of inserting all target chars
    for col in range(1, n+1):
        D[0, col] = col * ins_cost

    # Fill the rest
    for row in range(1, m+1):
        for col in range(1, n+1):
            r_cost = rep_cost
            if source[row-1] == target[col-1]:  # characters match → free
                r_cost = 0
            D[row, col] = min(
                D[row-1, col] + del_cost,      # delete
                D[row, col-1] + ins_cost,      # insert
                D[row-1, col-1] + r_cost       # replace or match
            )

    med = D[m, n]
    return D, med

source = 'play'
target = 'stay'
matrix, min_edits = min_edit_distance(source, target)
print(f"Minimum edits: {min_edits}")

# Display as a readable table
df = pd.DataFrame(matrix, index=list('#' + source), columns=list('#' + target))
print(df)

**Reading the matrix:** The bottom-right cell is always the final answer. Each cell was built from the three cells above it, to its left, and diagonally up-left.

Try changing `source` and `target` and re-run to see the matrix change.

---
## Quick Reference Card

| Pattern | Syntax | Use case |
|---------|--------|----------|
| Safe dict lookup | `d.get(key, 0)` | Avoid KeyError, default to 0 |
| Dict comprehension | `{k: f(v) for k, v in d.items()}` | Transform a dictionary |
| Split pattern | `[(w[:i], w[i:]) for i in range(len(w)+1)]` | All edit operations |
| Set union | `a \| b` | Combine two sets, drop duplicates |
| Set intersection | `a & b` | Keep only shared elements |
| Set union-assign | `a \|= b` | Merge b into a in place |
| Lambda | `lambda x: expression` | Quick one-line function |
| Sort by key | `sorted(lst, key=lambda x: ..., reverse=True)` | Custom sort order |
| Short-circuit or | `[] or ['fallback']` | Priority chains |